# Gene Expression Analysis under Oxidative Stress

## Introduction
This project analyzes how normal pancreatic cells (hTERT-HPNE) and tumor cells 
(PANC-1) respond to oxidative stress induced by H2O2, through differential gene 
expression analysis using RNA-seq data from study GSE196284 (Liu et al., Antioxidants, 2022).

## Dataset
- **Source:** Gene Expression Omnibus (GEO) — GSE196284
- **Organism:** Homo sapiens
- **Data type:** RNA-seq raw counts
- **Conditions:** Normal cells (HPNE) vs tumor cells (PANC-1), with and without H2O2 treatment

## Data Dictionary
| Sample | Cell type | Condition |
|--------|-----------|-----------|
| GSM5866903 | hTERT-HPNE (normal) | Control |
| GSM5866904 | hTERT-HPNE (normal) | H2O2 (oxidative stress) |
| GSM5866905 | PANC-1 (tumor) | Control |
| GSM5866906 | PANC-1 (tumor) | H2O2 (oxidative stress) |

## Objectives
1. Explore and understand the structure and quality of gene expression data
2. Identify differentially expressed genes between conditions
3. Build a classification model to predict cell type and condition
4. Deploy results as a REST API in the cloud

In [4]:
#Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [5]:
# Visualization settings

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data loading

In [6]:
# Dataset loading
df_counts_raw = pd.read_csv(
    '../data/GSE196284_raw_counts_GRCh38.p13_NCBI.tsv.gz',
    sep='\t',
    compression='gzip'
)

print(f"Dataset cargado: {df_counts_raw.shape[0]} genes x {df_counts_raw.shape[1]} columnas")
df_counts_raw.head()

Dataset cargado: 39376 genes x 5 columnas


,GeneID,GSM5866903,GSM5866904,GSM5866905,GSM5866906
0,100287102,4,7,21,14
1,653635,241,305,611,652
2,102466751,6,17,26,39
3,107985730,0,0,0,0
4,100302278,0,0,0,0


## 2. Column renaming

In [7]:
# Rename columns with meaningful names
df_counts_raw.rename(columns={
    'GSM5866903': 'HPNE_Control',
    'GSM5866904': 'HPNE_H2O2',
    'GSM5866905': 'PANC1_Control',
    'GSM5866906': 'PANC1_H2O2'
}, inplace=True)

df_counts_raw.head()

,GeneID,HPNE_Control,HPNE_H2O2,PANC1_Control,PANC1_H2O2
0,100287102,4,7,21,14
1,653635,241,305,611,652
2,102466751,6,17,26,39
3,107985730,0,0,0,0
4,100302278,0,0,0,0


## 3. Gene annotation

In [8]:
# Load gene annotation
df_annot = pd.read_csv(
    '../data/Human.GRCh38.p13.annot.tsv.gz',
    sep='\t',
    compression='gzip'
)

df_annot.head()

C:\Users\Federica\AppData\Local\Temp\ipykernel_20448\3128641179.py:2: DtypeWarning: Columns (0: ChrStart, 1: ChrStop) have mixed types. Specify dtype option on import or set low_memory=False.
  df_annot = pd.read_csv(


,GeneID,Symbol,Description,Synonyms,GeneType,EnsemblGeneID,Status,ChrAcc,ChrStart,ChrStop,Orientation,Length,GOFunctionID,GOProcessID,GOComponentID,GOFunction,GOProcess,GOComponent
0,100287102,DDX11L1,DEAD/H-box helicase 11 like 1 (pseudogene),NaN,pseudo,ENSG00000290825,active,NC_000001.11,11874,14409,positive,1652,NaN,NaN,NaN,NaN,NaN,NaN
1,653635,WASH7P,"WASP family homolog 7, pseudogene",FAM39F|WASH5P,pseudo,NaN,active,NC_000001.11,14362,29370,negative,1769,NaN,NaN,NaN,NaN,NaN,NaN
2,102466751,MIR6859-1,microRNA 6859-1,hsa-mir-6859-1,ncRNA,ENSG00000278267,active,NC_000001.11,17369,17436,negative,68,NaN,NaN,NaN,NaN,NaN,NaN
3,107985730,MIR1302-2HG,MIR1302-2 host gene,NaN,ncRNA,NaN,active,NC_000001.11,29926,31295,positive,538,NaN,NaN,NaN,NaN,NaN,NaN
4,100302278,MIR1302-2,microRNA 1302-2,MIRN1302-2|hsa-mir-1302-2,ncRNA,ENSG00000284332,active,NC_000001.11,30366,30503,positive,138,NaN,GO:0035195,NaN,NaN,miRNA-mediated gene silencing,NaN


In [9]:
# Merge counts with gene annotation
df_counts_raw = df_counts_raw.merge(
    df_annot[['GeneID', 'Symbol']],
    on='GeneID',
    how='left'
)

# Set gene symbol as index
df_counts_raw = df_counts_raw.set_index('Symbol').drop(columns=['GeneID'])

df_counts_raw.head()

,HPNE_Control,HPNE_H2O2,PANC1_Control,PANC1_H2O2
Symbol,,,,
DDX11L1,4,7,21,14
WASH7P,241,305,611,652
MIR6859-1,6,17,26,39
MIR1302-2HG,0,0,0,0
MIR1302-2,0,0,0,0


## 4. Initial Exploration

In [10]:
df_counts_raw.info()

<class 'pandas.DataFrame'>
Index: 39376 entries, DDX11L1 to TRNP
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   HPNE_Control   39376 non-null  int64
 1   HPNE_H2O2      39376 non-null  int64
 2   PANC1_Control  39376 non-null  int64
 3   PANC1_H2O2     39376 non-null  int64
dtypes: int64(4)
memory usage: 1.5+ MB


In [11]:
df_counts_raw.describe()

,HPNE_Control,HPNE_H2O2,PANC1_Control,PANC1_H2O2
count,3.937600e+04,3.937600e+04,39376.000000,39376.000000
mean,8.553432e+02,8.247941e+02,1334.672135,1146.625533
std,1.358934e+04,1.721197e+04,8416.512797,6903.697262
min,0.000000e+00,0.000000e+00,0.000000,0.000000
25%,0.000000e+00,0.000000e+00,0.000000,0.000000
50%,4.000000e+00,6.000000e+00,7.000000,6.000000
75%,2.930000e+02,2.740000e+02,461.250000,415.500000
max,1.701589e+06,2.331029e+06,751851.000000,611011.000000


## 5. Data Quality Check

In [13]:
# Check duplicates, nulls and zero-count genes
print("=== Data Quality Check ===\n")

# Duplicates
duplicates = df_counts_raw.index.duplicated().sum()
print(f"Duplicate gene symbols: {duplicates}")

# Null gene symbols
null_symbols = df_counts_raw.index.isna().sum()
print(f"Null gene symbols: {null_symbols}")

# Null values in counts
null_counts = df_counts_raw.isnull().sum().sum()
print(f"Null values in counts: {null_counts}")

# Zero-count genes
total_genes = len(df_counts_raw)
zero_genes = (df_counts_raw.sum(axis=1) == 0).sum()
print(f"\nTotal genes: {total_genes}")
print(f"Genes with zero counts in all samples: {zero_genes} ({zero_genes/total_genes*100:.1f}%)")
print(f"Genes with some expression: {total_genes - zero_genes}")

=== Data Quality Check ===

Duplicate gene symbols: 2
Null gene symbols: 0
Null values in counts: 0

Total genes: 39376
Genes with zero counts in all samples: 8438 (21.4%)
Genes with some expression: 30938


In [14]:
# Fix data quality issues

# 1. Remove duplicate gene symbols
df_counts_raw = df_counts_raw[~df_counts_raw.index.duplicated(keep='first')]
print(f"After removing duplicates: {len(df_counts_raw)} genes")

# 2. Remove genes with zero counts in all samples
df_counts_filtered = df_counts_raw[df_counts_raw.sum(axis=1) > 0]
print(f"After removing zero-count genes: {len(df_counts_filtered)} genes")
print(f"Genes removed: {len(df_counts_raw) - len(df_counts_filtered)}")

After removing duplicates: 39374 genes
After removing zero-count genes: 30936 genes
Genes removed: 8438
